In [ ]:
# Find the project root from either the workspace or notebooks folder.
from pathlib import Path
import sys
PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "src/task_b").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the NLP_PROJECT folder in VS Code before running this notebook")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Set official FEVER inputs; this notebook never downloads files.
from task_b.fever import prepare_fever, write_prepared, load_source_prior
from task_b.verification import majority_prior, lexical_overlap, lexical_cues, timed_predictions, evaluate, fever_binary_proxy, save_run
TRAIN_CLAIMS = PROJECT_ROOT / "data/raw/fever/train.jsonl"
EVAL_CLAIMS = PROJECT_ROOT / "data/raw/fever/shared_task_dev.jsonl"
WIKI_DIR = PROJECT_ROOT / "data/raw/fever/wiki-pages"
OUTPUT = PROJECT_ROOT / "outputs/task_b/fever_baselines"
if TRAIN_CLAIMS.resolve() == EVAL_CLAIMS.resolve():
    raise ValueError("Training prior and evaluation files must differ")
missing = [str(path) for path in (TRAIN_CLAIMS, EVAL_CLAIMS, WIKI_DIR)
           if not path.exists()]
if missing:
    raise FileNotFoundError("Provide the official FEVER inputs: " + ", ".join(missing))


In [ ]:
# Resolve gold evidence and inspect the resolution count.
print("Resolving FEVER evidence; the Wikipedia scan may take several minutes.")
records, report = prepare_fever(EVAL_CLAIMS, WIKI_DIR)
write_prepared(records, report, OUTPUT / "prepared")
print({key: report[key] for key in ("resolved", "unresolved", "skipped_nei")})
assert records, "No gold-evidence records resolved; inspect resolution_report.json"


In [ ]:
# Declare a prior from training labels; never use evaluation labels to fit it.
prior_labels = load_source_prior(TRAIN_CLAIMS)
prior = majority_prior(prior_labels)
print("Training prior:", prior, "from", TRAIN_CLAIMS)


In [ ]:
# Run fixed deterministic baselines on the same supplied evidence.
examples = [row.claim for row in records]
gold = [row.gold_label for row in records]
methods = {"majority": lambda _: prior, "lexical": lexical_overlap, "lexical_cues": lexical_cues}
results = {}
for name, predictor in methods.items():
    predictions, latency = timed_predictions(examples, predictor)
    binary = [fever_binary_proxy(label) for label in predictions]
    score = evaluate(gold, binary, ("SUPPORTED", "CONTRADICTED"), latency)
    score["raw_unsupported_count"] = predictions.count("UNSUPPORTED")
    score["evaluation_scope"] = "FEVER SUPPORTS/REFUTES gold-evidence diagnostic"
    results[name] = score
    save_run(OUTPUT / name, {"method": name, "prior_source": str(TRAIN_CLAIMS), "evaluation_source": str(EVAL_CLAIMS), "threshold": 0.5 if name != "majority" else None, "binary_proxy": "UNSUPPORTED -> CONTRADICTED for this diagnostic only"}, [{"claim_id": row.claim.claim_id, "gold": row.gold_label, "raw_prediction": prediction, "binary_prediction": projected, "latency_seconds": elapsed} for row, prediction, projected, elapsed in zip(records, predictions, binary, latency)], score)


In [ ]:
# Print compact scores and readable two-class confusion matrices.
print(f"{'method':<16} {'n':>7} {'accuracy':>10} {'macro-F1':>10} {'wrong support':>15} {'mean ms':>10} {'raw unsupported':>17}")
for name, score in results.items():
    print(f"{name:<16} {score['n']:>7} {score['accuracy']:>10.3f} {score['macro_f1']:>10.3f} {score['incorrect_supported']:>15} {score['latency_mean_ms']:>10.3f} {score['raw_unsupported_count']:>17}")

# Keep class scores and confusion matrices separate for each method.
for name, score in results.items():
    print("\n" + name + " per-class scores")
    print(f"{'class':<16} {'precision':>10} {'recall':>10} {'F1':>10} {'support':>10}")
    for label, values in score['per_class'].items():
        print(f"{label:<16} {values['precision']:>10.3f} {values['recall']:>10.3f} {values['f1']:>10.3f} {values['support']:>10}")
    print("confusion matrix (gold rows, predicted columns)")
    print(f"{'':<16} " + " ".join(f"{label:>14}" for label in score['labels']))
    for label, counts in zip(score['labels'], score['confusion_matrix']):
        print(f"{label:<16} " + " ".join(f"{count:>14}" for count in counts))
